# Train your first 🐸 TTS model 💫

### 👋 Hello and welcome to Coqui (🐸) TTS

The goal of this notebook is to show you a **typical workflow** for **training** and **testing** a TTS model with 🐸.

Let's train a very small model on a very small amount of data so we can iterate quickly.

In this notebook, we will:

1. Download data and format it for 🐸 TTS.
2. Configure the training and testing runs.
3. Train a new model.
4. Test the model and display its performance.

So, let's jump right in!


In [1]:
## Install Coqui TTS
! pip install -U pip
! pip install TTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 98.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 123.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 208.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... d

## ✅ Data Preparation

### **First things first**: we need some data.

We're training a Text-to-Speech model, so we need some _text_ and we need some _speech_. Specificially, we want _transcribed speech_. The speech must be divided into audio clips and each clip needs transcription. More details about data requirements such as recording characteristics, background noise and vocabulary coverage can be found in the [🐸TTS documentation](https://tts.readthedocs.io/en/latest/formatting_your_dataset.html).

If you have a single audio file and you need to **split** it into clips. It is also important to use a lossless audio file format to prevent compression artifacts. We recommend using **wav** file format.

The data format we will be adopting for this tutorial is taken from the widely-used  **LJSpeech** dataset, where **waves** are collected under a folder:

<span style="color:purple;font-size:15px">
/wavs<br />
 &emsp;| - audio1.wav<br />
 &emsp;| - audio2.wav<br />
 &emsp;| - audio3.wav<br />
  ...<br />
</span>

and a **metadata.csv** file will have the audio file name in parallel to the transcript, delimited by `|`:

<span style="color:purple;font-size:15px">
# metadata.csv <br />
audio1|This is my sentence. <br />
audio2|This is maybe my sentence. <br />
audio3|This is certainly my sentence. <br />
audio4|Let this be your sentence. <br />
...
</span>

In the end, we should have the following **folder structure**:

<span style="color:purple;font-size:15px">
/MyTTSDataset <br />
&emsp;| <br />
&emsp;| -> metadata.csv<br />
&emsp;| -> /wavs<br />
&emsp;&emsp;| -> audio1.wav<br />
&emsp;&emsp;| -> audio2.wav<br />
&emsp;&emsp;| ...<br />
</span>

🐸TTS already provides tooling for the _LJSpeech_. if you use the same format, you can start training your models right away. <br />

After you collect and format your dataset, you need to check two things. Whether you need a **_formatter_** and a **_text_cleaner_**. <br /> The **_formatter_** loads the text file (created above) as a list and the **_text_cleaner_** performs a sequence of text normalization operations that converts the raw text into the spoken representation (e.g. converting numbers to text, acronyms, and symbols to the spoken format).

If you use a different dataset format then the LJSpeech or the other public datasets that 🐸TTS supports, then you need to write your own **_formatter_** and  **_text_cleaner_**.

## ⏳️ Loading your dataset
Load one of the dataset supported by 🐸TTS.

We will start by defining dataset config and setting LJSpeech as our target dataset and define its path.


In [1]:
import os

# BaseDatasetConfig: defines name, formatter and path of the dataset.
from TTS.tts.configs.shared_configs import BaseDatasetConfig

output_path = "tts_train_dir"
if not os.path.exists(output_path):
    os.makedirs(output_path)


In [2]:
# Download and extract LJSpeech dataset.

!curl -L "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2" -o "$output_path/LJSpeech-1.1.tar.bz2"
!tar -xf $output_path/LJSpeech-1.1.tar.bz2 -C $output_path

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2621M  100 2621M    0     0   265M      0  0:00:09  0:00:09 --:--:--  239M


In [3]:
import os
import random

wav_folder = "/content/tts_train_dir/LJSpeech-1.1/wavs"

# Get all wav files in the folder
all_wav_files = [f for f in os.listdir(wav_folder) if f.endswith('.wav')]

print(f"Total WAV files before deletion: {len(all_wav_files)}")

# Calculate how many files to delete (approximately half)
num_files_to_delete = len(all_wav_files) // 2

# Randomly select files to delete
files_to_delete = random.sample(all_wav_files, num_files_to_delete)

# Delete the selected files
for file_name in files_to_delete:
    file_path = os.path.join(wav_folder, file_name)
    os.remove(file_path)
    print(f"Deleted: {file_name}")

# Verify remaining files
remaining_wav_files = [f for f in os.listdir(wav_folder) if f.endswith('.wav')]
print(f"Total WAV files after deletion: {len(remaining_wav_files)}")

Streaming output truncated to the last 5000 lines.
Deleted: LJ001-0005.wav
Deleted: LJ002-0053.wav
Deleted: LJ019-0239.wav
Deleted: LJ004-0246.wav
Deleted: LJ033-0126.wav
Deleted: LJ004-0047.wav
Deleted: LJ008-0308.wav
Deleted: LJ016-0193.wav
Deleted: LJ018-0062.wav
Deleted: LJ015-0210.wav
Deleted: LJ038-0189.wav
Deleted: LJ040-0074.wav
Deleted: LJ002-0046.wav
Deleted: LJ009-0285.wav
Deleted: LJ045-0008.wav
Deleted: LJ039-0116.wav
Deleted: LJ026-0062.wav
Deleted: LJ012-0094.wav
Deleted: LJ015-0215.wav
Deleted: LJ043-0050.wav
Deleted: LJ049-0112.wav
Deleted: LJ003-0201.wav
Deleted: LJ024-0109.wav
Deleted: LJ028-0273.wav
Deleted: LJ019-0294.wav
Deleted: LJ040-0101.wav
Deleted: LJ015-0226.wav
Deleted: LJ018-0114.wav
Deleted: LJ016-0221.wav
Deleted: LJ049-0142.wav
Deleted: LJ050-0033.wav
Deleted: LJ009-0091.wav
Deleted: LJ003-0121.wav
Deleted: LJ045-0024.wav
Deleted: LJ037-0166.wav
Deleted: LJ014-0052.wav
Deleted: LJ041-0055.wav
Deleted: LJ038-0071.wav
Deleted: LJ023-0093.wav
Deleted: LJ03

In [4]:
dataset_config = BaseDatasetConfig(
    formatter="ljspeech", meta_file_train="metadata.csv", path=os.path.join(output_path, "LJSpeech-1.1/")
)

## ✅ Train a new model

Let's kick off a training run 🚀🚀🚀.

Deciding on the model architecture you'd want to use is based on your needs and available resources. Each model architecture has it's pros and cons that define the run-time efficiency and the voice quality.
We have many recipes under `TTS/recipes/` that provide a good starting point. For this tutorial, we will be using `GlowTTS`.

We will begin by initializing the model training configuration.

In [5]:
# GlowTTSConfig: all model related values for training, validating and testing.
from TTS.tts.configs.glow_tts_config import GlowTTSConfig
config = GlowTTSConfig(
    batch_size=32,
    eval_batch_size=16,
    num_loader_workers=2,
    num_eval_loader_workers=2,
    run_eval=True,
    test_delay_epochs=-1,
    epochs=40,
    text_cleaner="phoneme_cleaners",
    use_phonemes=True,
    phoneme_language="en-us",
    phoneme_cache_path=os.path.join(output_path, "phoneme_cache"),
    print_step=25,
    print_eval=False,
    mixed_precision=True,
    output_path=output_path,
    datasets=[dataset_config],
    save_step=1000,
)

Next we will initialize the audio processor which is used for feature extraction and audio I/O.

In [6]:
from TTS.utils.audio import AudioProcessor
ap = AudioProcessor.init_from_config(config)
# Modify sample rate if for a custom audio dataset:
# ap.sample_rate = 22050


 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:True
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:45
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024


Next we will initialize the tokenizer which is used to convert text to sequences of token IDs.  If characters are not defined in the config, default characters are passed to the config.

In [7]:
from TTS.tts.utils.text.tokenizer import TTSTokenizer
tokenizer, config = TTSTokenizer.init_from_config(config)

Next we will load data samples. Each sample is a list of ```[text, audio_file_path, speaker_name]```. You can define your custom sample loader returning the list of samples.

In [ ]:
from TTS.tts.datasets import load_tts_samples
import os

# Load the initial samples
initial_train_samples, initial_eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

# I added code here to use the downaloded dataset and filter out any samples whose audio files do not exist, which is likely the case for the deleted files. This way, we can ensure that the training and evaluation processes only use valid samples with existing audio files.
train_samples = [sample for sample in initial_train_samples if os.path.exists(sample['audio_file'])]
eval_samples = [sample for sample in initial_eval_samples if os.path.exists(sample['audio_file'])]

print(f"Initial training samples loaded: {len(initial_train_samples)}")
print(f"Training samples after filtering for existing audio files: {len(train_samples)}")
print(f"Initial evaluation samples loaded: {len(initial_eval_samples)}")
print(f"Evaluation samples after filtering for existing audio files: {len(eval_samples)}")

 | > Found 13100 files in /content/tts_train_dir/LJSpeech-1.1
Initial training samples loaded: 12969
Training samples after filtering for existing audio files: 6481
Initial evaluation samples loaded: 131
Evaluation samples after filtering for existing audio files: 69


Now we're ready to initialize the model.

Models take a config object and a speaker manager as input. Config defines the details of the model like the number of layers, the size of the embedding, etc. Speaker manager is used by multi-speaker models.

In [9]:
from TTS.tts.models.glow_tts import GlowTTS
model = GlowTTS(config, ap, tokenizer, speaker_manager=None)

Trainer provides a generic API to train all the 🐸TTS models with all its perks like mixed-precision training, distributed training, etc.

In [10]:
from trainer import Trainer, TrainerArgs
trainer = Trainer(
    TrainerArgs(), config, output_path, model=model, train_samples=train_samples, eval_samples=eval_samples
)

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 12
 | > Num. of Torch Threads: 6
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=tts_train_dir/run-March-28-2026_11+08AM-0000000
/usr/local/lib/python3.11/dist-packages/trainer/trainer.py:552: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()

 > Model has 28610257 parameters


### AND... 3,2,1... START TRAINING 🚀🚀🚀

In [11]:
trainer.fit()


 > EPOCH: 0/40
 --> tts_train_dir/run-March-28-2026_11+08AM-0000000


[*] Pre-computing phonemes...


  0%|          | 5/6481 [00:00<17:49,  6.06it/s]

ə t͡ʃeɪn fɪkst tə steɪpəlz æt hɪz bæk pæst ɹaʊnd hɪz t͡ʃɛst ʌndɚ hɪz ɑɹmz, ænd wəz pædlɑkt ɔn ðə lɛft saɪd,
 [!] Character '͡' not found in the vocabulary. Discarding it.


 16%|█▌        | 1042/6481 [00:48<03:26, 26.31it/s]

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.005855858325958252 (+0.0018230676651000977)
     | > avg_loss: 0.2747260332107544 (-0.0309358611702919)
     | > avg_log_mle: -0.13411647081375122 (-0.014767825603485107)
     | > avg_loss_dur: 0.4088425040245056 (-0.016168035566806793)

 > BEST MODEL : tts_train_dir/run-March-28-2026_11+08AM-0000000/best_model_7714.pth

 > EPOCH: 38/40
 --> tts_train_dir/run-March-28-2026_11+08AM-0000000

 > TRAINING (2026-03-28 12:19:42) 

   --> TIME: 2026-03-28 12:19:47 -- STEP: 11/203 -- GLOBAL_STEP: 7725
     | > loss: 0.30578309297561646  (0.3027791001579978)
     | > log_mle: -0.09435892105102539  (-0.10044039379466664)
     | > loss_dur: 0.40014201402664185  (0.4032194939526645)
     | > amp_scaler: 32768.0  (32768.0)
     | > grad_norm: tensor(2.5562, device='cuda:0')  (tensor(2.0165, device='cuda:0'))
     | > current_lr: 9.499999999999999e-06 
     | > step_time: 0.319  (0.3439665057442405)
     | > loader_time: 0.0034  (0.0030235593969171

 | > Synthesizing test sentences.



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.003754913806915283 (-0.0021009445190429688)
     | > avg_loss: 0.2514253780245781 (-0.0233006551861763)
     | > avg_log_mle: -0.13957130908966064 (-0.005454838275909424)
     | > avg_loss_dur: 0.39099668711423874 (-0.017845816910266876)

 > BEST MODEL : tts_train_dir/run-March-28-2026_11+08AM-0000000/best_model_7917.pth

 > EPOCH: 39/40
 --> tts_train_dir/run-March-28-2026_11+08AM-0000000

 > TRAINING (2026-03-28 12:21:34) 

   --> TIME: 2026-03-28 12:21:37 -- STEP: 8/203 -- GLOBAL_STEP: 7925
     | > loss: 0.2962927222251892  (0.28204765170812607)
     | > log_mle: -0.11339855194091797  (-0.11026613414287567)
     | > loss_dur: 0.4096912741661072  (0.39231378585100174)
     | > amp_scaler: 32768.0  (32768.0)
     | > grad_norm: tensor(1.9536, device='cuda:0')  (tensor(2.0765, device='cuda:0'))
     | > current_lr: 9.75e-06 
     | > step_time: 0.3215  (0.3135277032852173)
     | > loader_time: 0.0032  (0.0030360519886016846)


   --

 | > Synthesizing test sentences.



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.003685176372528076 (-6.973743438720703e-05)
     | > avg_loss: 0.22168872505426407 (-0.029736652970314026)
     | > avg_log_mle: -0.14957216382026672 (-0.010000854730606079)
     | > avg_loss_dur: 0.3712608888745308 (-0.019735798239707947)

 > BEST MODEL : tts_train_dir/run-March-28-2026_11+08AM-0000000/best_model_8120.pth


#### 🚀 Run the Tensorboard. 🚀
On the notebook and Tensorboard, you can monitor the progress of your model. Also Tensorboard provides certain figures and sample outputs.

In [25]:
!pip install tensorboard
!tensorboard --logdir=tts_train_dir

2026-03-28 12:54:11.458980: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774702451.480352   32720 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774702451.486814   32720 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.18.0 at http://localhost:6006/ (Press CTRL+C to quit)
Traceback (most recent call last):
  File "/usr/local/bin/tensorboard", line 10, in 

## ✅ Test the model

We made it! 🙌

Let's kick off the testing run, which displays performance metrics.

We're committing the cardinal sin of ML 😈 (aka - testing on our training data) so you don't want to deploy this model into production. In this notebook we're focusing on the workflow itself, so it's forgivable 😇

You can see from the test output that our tiny model has overfit to the data, and basically memorized this one sentence.

When you start training your own models, make sure your testing data doesn't include your training data 😅

Let's get the latest saved checkpoint.

In [ ]:
import glob, os
output_path = "tts_train_dir"
test_ckpt = sorted([f for f in glob.glob(output_path+"/*/*.pth")])[0] # I selected a particular model as opposed to returning the list of models.
print(test_ckpt)
test_config = sorted([f for f in glob.glob(output_path+"/*/*.json")])[0]

tts_train_dir/run-March-28-2026_11+08AM-0000000/best_model.pth


In [36]:
!tts --text "Assignment 7 for TTS" \
      --model_path $test_ckpt \
      --config_path $test_config \
      --out_path out.wav

 > Using model: glow_tts
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:True
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:45
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Text: Assignment 7 for TTS
 > Text splitted to sentences.
['Assignment 7 for TTS']
 > Processing time: 0.9543046951293945
 > Real-time factor: 0.4235591491063436
 > Saving output to out.wav


## 📣 Listen to the synthesized wave 📣

In [ ]:
import IPython
IPython.display.Audio("out.wav") 

## The output audio file is not very clear, but I was able to generate it using the trained model.
## I suspect this is due to reducing the number of training samples by half, which may have impacted the model's ability to learn effectively as well as significantly reducing the number of epochs.

## 🎉 Congratulations! 🎉 You now have trained your first TTS model!
Follow up with the next tutorials to learn more advanced material.